# 03 Final comparison of the six models

Runs on CPU. Input: the dataset with the folders `resultados/<model>/` of the six models.
Output in `/kaggle/working/final/`: `global_metrics.csv`, `per_class_metrics.csv`, `ordinal_metrics.csv`, `mcnemar.csv`,
`ensembles.csv`, `mcnemar_ensembles.csv` and the 12 figures (`fig_cm_<model>.png`, `fig_curves_<model>.png`) with the same style.
The rules of the ensembles are fixed before looking at the test subset, and their members are chosen with the validation subset.

In [ ]:
# Cell 1: load the final version of each model (the one with 150 test patches)
import os, glob, json, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import confusion_matrix, cohen_kappa_score, f1_score, precision_recall_fscore_support
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
INPUT, OUT = "/kaggle/input", "/kaggle/working/final"; os.makedirs(OUT, exist_ok=True)
MODELS = ["resnet50", "efficientnet_b0", "vit_b16", "swin_t", "cnn_lstm", "convlstm"]
NOMBRE = {"resnet50": "ResNet50", "efficientnet_b0": "EfficientNetB0", "vit_b16": "ViT-B/16",
          "swin_t": "Swin-T", "cnn_lstm": "CNN+LSTM", "convlstm": "ConvLSTM"}
CLASES = ["No deforestation", "Low", "Moderate", "Severe"]
P, M, H = {}, {}, {}
for m in MODELS:
    cand = [f for f in glob.glob(f"{INPUT}/**/pred_test_{m}.csv", recursive=True) if len(pd.read_csv(f)) == 150]
    assert len(cand) >= 1, f"no pred_test_{m}.csv with 150 patches"
    if len(cand) > 1: print("warning,", m, "has", len(cand), "copies with 150 patches; using", cand[0])
    d = os.path.dirname(cand[0])
    P[m] = pd.read_csv(cand[0]).set_index("patch_id").sort_index()
    M[m] = json.load(open(f"{d}/metrics_{m}.json")); H[m] = pd.read_csv(f"{d}/history_{m}.csv")
    print(f"{m:16s} {cand[0]}")
ref = P[MODELS[0]]
for m in MODELS:
    assert (P[m].index == ref.index).all() and (P[m].y_true == ref.y_true).all(), f"{m} does not use the same test subset"
print("the six models use the same 150 test patches")

In [ ]:
# Cell 2: global table (accuracy, 95% Wilson CI, macro F1, parameters and times)
filas = []
for m in MODELS:
    y, yh = P[m].y_true.values, P[m].y_pred.values; k = int((y == yh).sum())
    lo, hi = proportion_confint(k, len(y), alpha=.05, method="wilson")
    h = H[m]; b = int(h.val_loss.idxmin())
    filas.append(dict(model=NOMBRE[m], accuracy=k / len(y), ci95_low=lo, ci95_high=hi,
                      f1_macro=f1_score(y, yh, average="macro"), params=M[m].get("params"),
                      epochs_run=len(h), best_epoch=int(h.epoch[b]), train_minutes=M[m].get("train_minutes"),
                      inference_ms_per_patch=M[m].get("inference_ms_per_patch")))
TG = pd.DataFrame(filas).sort_values("accuracy", ascending=False)
TG.to_csv(f"{OUT}/global_metrics.csv", index=False); print(TG.round(4).to_string(index=False))
for m in MODELS:                                         # check: must match the metrics.json of each model
    if abs(M[m]["accuracy"] - (P[m].y_true == P[m].y_pred).mean()) > 1e-9: print("warning, mismatch:", m)

In [ ]:
# Cell 3: precision, recall and F1-score per class
filas = []
for m in MODELS:
    pr, rc, f1, n = precision_recall_fscore_support(P[m].y_true, P[m].y_pred, labels=[0, 1, 2, 3], zero_division=0)
    for c in range(4):
        filas.append(dict(model=NOMBRE[m], **{"class": CLASES[c]}, n=int(n[c]), precision=pr[c], recall=rc[c], f1=f1[c]))
TC = pd.DataFrame(filas); TC.to_csv(f"{OUT}/per_class_metrics.csv", index=False)
print(TC.pivot(index="model", columns="class", values="recall")[CLASES].round(3).to_string())

In [ ]:
# Cell 4: ordinal metrics and grouped classes
a3 = {0: 0, 1: 1, 2: 1, 3: 2}; a2 = {0: 0, 1: 1, 2: 1, 3: 1}
filas = []
for m in MODELS:
    y, yh = P[m].y_true, P[m].y_pred; kb = int((y.map(a2) == yh.map(a2)).sum())
    lo, hi = proportion_confint(kb, len(y), alpha=.05, method="wilson")
    filas.append(dict(model=NOMBRE[m], acc_4_classes=(y == yh).mean(), acc_within_1=((y - yh).abs() <= 1).mean(),
                      mae_classes=(y - yh).abs().mean(), quadratic_kappa=cohen_kappa_score(y, yh, weights="quadratic"),
                      acc_3_classes=(y.map(a3) == yh.map(a3)).mean(), acc_binary=kb / len(y),
                      binary_ci95_low=lo, binary_ci95_high=hi))
TO = pd.DataFrame(filas).sort_values("acc_4_classes", ascending=False)
TO.to_csv(f"{OUT}/ordinal_metrics.csv", index=False); print(TO.round(3).to_string(index=False))
print("3 groups: no deforestation | low + moderate | severe. Binary: no loss vs loss.")

In [ ]:
# Cell 5: exact McNemar test between the 15 pairs of models
filas = []
for a, b in itertools.combinations(MODELS, 2):
    ca, cb = P[a].y_pred == P[a].y_true, P[b].y_pred == P[b].y_true
    t = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
    r = mcnemar(t, exact=True)
    filas.append(dict(model_a=NOMBRE[a], model_b=NOMBRE[b], acc_a=ca.mean(), acc_b=cb.mean(),
                      only_a_correct=t[0][1], only_b_correct=t[1][0], p_value=r.pvalue, significant=r.pvalue < .05,
                      significant_bonferroni=r.pvalue < .05 / 15))
TM = pd.DataFrame(filas); TM.to_csv(f"{OUT}/mcnemar.csv", index=False)
print(TM.round(4).to_string(index=False)); print("pairs below 0.05:", int(TM.significant.sum()), "| below 0.0033:", int(TM.significant_bonferroni.sum()))

In [ ]:
# Cell 6: the 12 figures with the same style
for m in MODELS:
    cm = confusion_matrix(P[m].y_true, P[m].y_pred, labels=[0, 1, 2, 3]); plt.figure(figsize=(5.2, 4.4))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", xticklabels=CLASES, yticklabels=CLASES)
    plt.xlabel("Predicted class"); plt.ylabel("Reference class"); plt.tight_layout()
    plt.savefig(f"{OUT}/fig_cm_{m}.png", dpi=300); plt.close()
    h = H[m]; fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot(h.epoch, h.train_loss, label="Training"); ax[0].plot(h.epoch, h.val_loss, label="Validation")
    ax[1].plot(h.epoch, h.train_acc, label="Training"); ax[1].plot(h.epoch, h.val_acc, label="Validation")
    for a in ax:
        a.set_xlabel("Epoch"); a.legend()
        a.xaxis.set_major_locator(MultipleLocator(2 if len(h) <= 25 else 10))   # integer epochs: 2, 4, 6... (10, 20... with 50 epochs)
    ax[0].set_ylabel("Loss"); ax[1].set_ylabel("Accuracy")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_curves_{m}.png", dpi=300); plt.close()
print("figures:", len([f for f in os.listdir(OUT) if f.endswith(".png")]), "(must be 12)")

In [ ]:
# Cell 7: soft voting ensembles (rules fixed BEFORE looking at the test subset; members chosen with the validation subset)
FAM = {"resnet50": "CNN", "efficientnet_b0": "CNN", "vit_b16": "Transformer", "swin_t": "Transformer",
       "cnn_lstm": "Spatiotemporal", "convlstm": "Spatiotemporal"}
val_acc = {m: float(H[m].val_acc[H[m].val_loss.idxmin()]) for m in MODELS}     # validation accuracy of the final weights
print("validation accuracy of each model:", {NOMBRE[m]: round(v, 3) for m, v in val_acc.items()})
top3 = sorted(MODELS, key=lambda m: -val_acc[m])[:3]
por_familia = [max([m for m in MODELS if FAM[m] == f], key=lambda m: val_acc[m]) for f in ["CNN", "Transformer", "Spatiotemporal"]]
ENS = {"E1 average of the six models (main)": MODELS, "E2 three highest in validation": top3, "E3 highest in validation per family": por_familia}
pc = [f"p{k}" for k in range(4)]
y = ref.y_true; a2 = {0: 0, 1: 1, 2: 1, 3: 1}
filas, PE = [], {}
for nombre, miembros in ENS.items():
    prob = sum(P[m][pc].values for m in miembros) / len(miembros); yh = pd.Series(prob.argmax(1), index=ref.index)
    PE[nombre] = yh; k = int((yh == y).sum()); lo, hi = proportion_confint(k, len(y), alpha=.05, method="wilson")
    filas.append(dict(ensemble=nombre, members=", ".join(NOMBRE[m] for m in miembros), accuracy=k / len(y), ci95_low=lo,
                      ci95_high=hi, f1_macro=f1_score(y, yh, average="macro"),
                      quadratic_kappa=cohen_kappa_score(y, yh, weights="quadratic"), acc_binary=(y.map(a2) == yh.map(a2)).mean(),
                      **{f"recall_{c}": (yh[y == c] == c).mean() for c in range(4)}))
TE = pd.DataFrame(filas); TE.to_csv(f"{OUT}/ensembles.csv", index=False); print(TE.round(3).to_string(index=False))
filas = []
for nombre, yh in PE.items():                          # McNemar of each ensemble against each single model
    ce = yh == y
    for m in MODELS:
        cm_ = P[m].y_pred == y
        t = [[int((ce & cm_).sum()), int((ce & ~cm_).sum())], [int((~ce & cm_).sum()), int((~ce & ~cm_).sum())]]
        filas.append(dict(ensemble=nombre, model=NOMBRE[m], acc_ensemble=ce.mean(), acc_model=cm_.mean(),
                          only_ensemble_correct=t[0][1], only_model_correct=t[1][0], p_value=mcnemar(t, exact=True).pvalue))
TME = pd.DataFrame(filas); TME.to_csv(f"{OUT}/mcnemar_ensembles.csv", index=False)
print(TME.round(4).to_string(index=False))